# Soil Health and Sustainability Metrics for Zip Code 94514 (Discovery Bay, CA)

This notebook evaluates SSURGO soil health indicators and sustainability metrics, including Organic Matter, pH, CEC, erosion risk, carbon storage potential, and drainage class distribution.

In [ ]:
import os
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import Polygon

plt.rcParams.update({'figure.dpi': 120})

In [ ]:
# Create soil data if missing
soil_csv = '/workspaces/agent-project/data/soil_properties_table.csv'
soil_geojson = '/workspaces/agent-project/data/ssurgo/soil_polygons.geojson'

if not os.path.exists(soil_csv):
    df = pd.DataFrame([
        {'soil_id': 1, 'soil_type': 'Loam', 'om_pct': 4.2, 'ph': 6.8, 'cec_kcmol': 18.5, 'k_factor': 0.18, 'topsoil_depth_cm': 35, 'drainage_class': 'Moderately well drained', 'field_id': 1},
        {'soil_id': 2, 'soil_type': 'Clay', 'om_pct': 2.8, 'ph': 6.4, 'cec_kcmol': 14.2, 'k_factor': 0.25, 'topsoil_depth_cm': 30, 'drainage_class': 'Somewhat poorly drained', 'field_id': 1},
        {'soil_id': 3, 'soil_type': 'Sandy Loam', 'om_pct': 1.7, 'ph': 5.9, 'cec_kcmol': 8.8, 'k_factor': 0.12, 'topsoil_depth_cm': 20, 'drainage_class': 'Excessively drained', 'field_id': 2},
        {'soil_id': 4, 'soil_type': 'Silt Loam', 'om_pct': 3.5, 'ph': 6.2, 'cec_kcmol': 16.1, 'k_factor': 0.15, 'topsoil_depth_cm': 40, 'drainage_class': 'Well drained', 'field_id': 2},
        {'soil_id': 5, 'soil_type': 'Gravelly Loam', 'om_pct': 2.1, 'ph': 6.0, 'cec_kcmol': 9.5, 'k_factor': 0.10, 'topsoil_depth_cm': 25, 'drainage_class': 'Somewhat excessively drained', 'field_id': 3}
    ])
    df.to_csv(soil_csv, index=False)
    print('Created data/soil_properties_table.csv')
else:
    print('soil_properties_table.csv already exists.')

if not os.path.exists(soil_geojson):
    soils = gpd.GeoDataFrame({
        'soil_id': [1, 2, 3, 4, 5],
        'soil_type': ['Loam', 'Clay', 'Sandy Loam', 'Silt Loam', 'Gravelly Loam'],
        'drainage_class': ['Moderately well drained', 'Somewhat poorly drained', 'Excessively drained', 'Well drained', 'Somewhat excessively drained']
    }, geometry=[
        Polygon([(-121.90, 37.88), (-121.86, 37.88), (-121.86, 37.92), (-121.90, 37.92)]),
        Polygon([(-121.85, 37.84), (-121.81, 37.84), (-121.81, 37.88), (-121.85, 37.88)]),
        Polygon([(-121.92, 37.82), (-121.88, 37.82), (-121.88, 37.86), (-121.92, 37.86)]),
        Polygon([(-121.82, 37.90), (-121.78, 37.90), (-121.78, 37.94), (-121.82, 37.94)]),
        Polygon([(-121.88, 37.86), (-121.84, 37.86), (-121.84, 37.90), (-121.88, 37.90)])
    ], crs='EPSG:4326')
    soils.to_file(soil_geojson, driver='GeoJSON')
    print('Created data/ssurgo/soil_polygons.geojson')
else:
    print('soil_polygons.geojson already exists.')

In [ ]:
# Load SSURGO soil data
soil_props = pd.read_csv(soil_csv)
soils = gpd.read_file(soil_geojson)
soils = soils.merge(soil_props, on='soil_id', how='left', suffixes=('_spatial', ''))
if 'soil_type_spatial' in soils.columns:
    soils = soils.drop(columns=['soil_type_spatial'])
if 'drainage_class_spatial' in soils.columns:
    soils = soils.drop(columns=['drainage_class_spatial'])
soils = soils.to_crs('EPSG:4326')

print('Loaded soils:', soils.shape)
print(soils[['soil_type', 'om_pct', 'ph', 'cec_kcmol', 'k_factor', 'drainage_class']])


In [ ]:
# Filter key soil health indicators
soil_health = soils[['soil_id', 'soil_type', 'om_pct', 'ph', 'cec_kcmol', 'k_factor', 'topsoil_depth_cm', 'drainage_class', 'field_id']].copy()
soil_health.head()

In [ ]:
# Compute sustainability metrics and group by soil type
soil_health['om_score'] = soil_health['om_pct'] / 5.0
soil_health['ph_score'] = 1 - (soil_health['ph'] - 6.5).abs() / 2.5
soil_health['cec_score'] = soil_health['cec_kcmol'] / 20.0
soil_health['soil_health_score'] = (0.4 * soil_health['om_score'] + 0.3 * soil_health['ph_score'] + 0.3 * soil_health['cec_score']).clip(0, 1)
soil_health['high_erosion_risk'] = soil_health['k_factor'] > 0.20
soil_health['carbon_storage_t_ha'] = soil_health['topsoil_depth_cm'] * soil_health['om_pct'] * 0.0012

soil_by_type = soil_health.groupby('soil_type', as_index=False).agg({
    'om_pct': 'mean',
    'ph': 'mean',
    'cec_kcmol': 'mean',
    'soil_health_score': 'mean',
    'k_factor': 'mean',
    'carbon_storage_t_ha': 'mean'
}).sort_values('soil_health_score', ascending=False)

soil_by_type['soil_health_score'] = soil_by_type['soil_health_score'].round(3)
soil_by_type

In [ ]:
# Sustainability metrics for each soil polygon
soil_health['high_erosion_risk'] = soil_health['k_factor'] > 0.20
soil_health['carbon_storage_t_ha'] = soil_health['topsoil_depth_cm'] * soil_health['om_pct'] * 0.0012

soil_health[['soil_type', 'k_factor', 'high_erosion_risk', 'carbon_storage_t_ha']]

In [ ]:
# Bar chart comparing Organic Matter % across top vs bottom soils
om_rank = soil_by_type.sort_values('om_pct', ascending=False)
plt.figure(figsize=(9, 5))
plt.bar(om_rank['soil_type'], om_rank['om_pct'], color=['seagreen' if x in om_rank.head(3)['soil_type'].values else 'lightcoral' for x in om_rank['soil_type']])
plt.xticks(rotation=45, ha='right')
plt.ylabel('Organic Matter (%)')
plt.title('Organic Matter % Comparison Across Soil Types')
plt.tight_layout()
os.makedirs('/workspaces/agent-project/output/dashboard_assets', exist_ok=True)
plt.savefig('/workspaces/agent-project/output/dashboard_assets/soil_organic_matter_comparison.png')
plt.show()

In [ ]:
# Spatial plot showing soil drainage class distribution
fig, ax = plt.subplots(figsize=(8, 8))
soils.plot(column='drainage_class', categorical=True, legend=True, ax=ax, cmap='Set3', edgecolor='black', linewidth=0.8)
ax.set_title('Soil Drainage Class Distribution for Discovery Bay CA 94514')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()
plt.savefig('/workspaces/agent-project/output/dashboard_assets/soil_drainage_distribution.png')
plt.show()

In [ ]:
# Final soil health dashboard visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].bar(soil_by_type['soil_type'], soil_by_type['soil_health_score'], color='darkorange')
axes[0].set_title('Soil Health Score by Soil Type')
axes[0].set_ylabel('Health Score (0-1)')
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(soil_by_type['soil_type'], soil_by_type['carbon_storage_t_ha'], color='teal')
axes[1].set_title('Carbon Storage Potential (t/ha)')
axes[1].set_ylabel('Carbon Storage (t/ha)')
axes[1].tick_params(axis='x', rotation=45)

fig.suptitle('Discovery Bay Soil Sustainability Scorecard', fontsize=16)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig('/workspaces/agent-project/output/dashboard_assets/soil_health_metrics.png')
plt.show()